# <font color="Green">**Notebook Purpose**</font>

This notebook documents and illustrates the clustering analysis performed on patient-level medication trajectories. Using the trajectory representations generated in earlier preprocessing notebooks (e.g., `MedicationTrajectoryRepresentations.ipynb`), it:

- Fits clustering models to the patient-level vectors.
- Provides an overview of how the number of clusters (*k*) was selected, including examples of the metrics used (e.g., WCSS elbow plots, Silhouette scores, Gap Statistic), without reproducing all exploratory tuning code.
- Generates a final set of cluster assignments for each patient and creates groupings of the clusters based on overarching treatment patterns.


---

### <font color="Red">Required Data</font>

To run this notebook, you will need the following input object, generated in earlier preprocessing steps:

1. **Patient-level trajectory representation**
   - `patient_vectors.pkl`
     - A pickled Python dictionary mapping each `patient_id` to its corresponding fixed-length trajectory vector (e.g., the concatenated one-hot encodings of medication classes across the 12 half-year bins from 2019–2024).




In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

from joblib import Parallel, delayed # This specifically is used for the gap statistic, which takes a long time to compute
from tqdm import tqdm # Used as a progress bar for longer operations

##<font color="black">**Read in Data**</font>

In [ ]:
with open("/content/patient_vectors.pkl", "rb") as f:
    patient_vectors = pickle.load(f)

##<font color="black">**Clustering**</font>

This section will consist of two sub sections, with the overarching goal of producing multiple dictionaries that represents the clusters and the patients in each cluster.

The first sub-section will focus on choosing the optimal number of clusters (k).

The second sub-section will involve performing the final clustering using the determined k, and producing the aforementioned dictionaries of patient clusters.

### <font color="black">**Overview of Methods for Selecting k**</font>

In this section, rather than running the full set of clustering-validation procedures, I provide a brief overview of the common statistical approaches used to select an appropriate number of clusters (k) for patient medication trajectory data. To keep the supplementary materials concise, I include descriptions and representative examples of each method rather than the full exploratory code used during the development process.

The three primary approaches used in the analysis are:

**`WCSS Elbow Method:`** Within-Cluster Sum of Squares (WCSS) measures the compactness of clusters by summing the squared distances between each point and its cluster centroid. Because WCSS decreases as k increases, the “elbow” point—where the rate of decrease begins to level off—serves as a heuristic indicator for the optimal number of clusters. A plot of WCSS versus k typically reveals this inflection point, which we use as a visual guide.

**`Silhouette Score:`** The Silhouette Score evaluates how well each point fits within its assigned cluster compared to others by comparing intra-cluster and nearest-neighbor distances. Scores range from -1 to 1, with higher values indicating better-defined cluster structure. By computing the average score across all points for a range of k, one can identify the value that yields the most cohesive and well-separated clusters.

**`Gap Statistic:`** The Gap Statistic compares the clustering performance of the observed data to that of a reference dataset generated under a null (often uniform) distribution. For each k, the method measures the difference between the log(WCSS) of the real data and the expected log(WCSS) under the reference distribution. The optimal k is typically the smallest value within one standard deviation of the maximal gap, providing a more formal baseline-informed estimate.

In [ ]:
# Key data objects used in the analysis
k_values = list(range(2, 50))
patient_ids = list(patient_vectors.keys())
X = np.array([patient_vectors[pid] for pid in patient_ids])

####<font color="black">**WCSS Elbow Method**</font>

In [ ]:
def compute_wcss(X, labels):
    wcss = 0
    for cluster_id in np.unique(labels):
        cluster_points = X[labels == cluster_id]
        centroid = cluster_points.mean(axis=0)
        wcss += np.sum((cluster_points - centroid) ** 2)
    return wcss

In [ ]:
# For every k value, perform a clustering and track WCSS scores.
WCSS_scores = []

for k in k_values:
    clustering = AgglomerativeClustering(
       n_clusters=k,
       linkage='ward',
       metric='euclidean'
    )

    clustering.fit(X)
    labels = clustering.labels_
    wcss = compute_wcss(X, labels)
    WCSS_scores.append(wcss)

# Following this step, plot the WCSS scores per k and look for the 'elbow' in the plot.

####<font color="black">**Silhouette Score**</font>

In [ ]:
# For every k value, perform a clustering and track slihouette scores.
silhouette_scores = []

for k in k_values:
    clustering = AgglomerativeClustering(
        n_clusters=k,
        linkage='ward',
        metric='euclidean'
    )

    clustering.fit(X)
    labels = clustering.labels_
    silhouette_avg = silhouette_score(X, labels)
    silhouette_scores.append(silhouette_avg)

# Following this step, plot the silhouette scores per k and look for the highest scores.

####<font color="black">**Gap Statistic**</font>

In [ ]:
def generate_reference_data(X, B):
    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    return [np.random.uniform(low=mins, high=maxs, size=X.shape) for _ in range(B)]

In [ ]:
def cluster_reference_data(k, ref_data, b_idx):
    model_ref = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels_ref = model_ref.fit_predict(ref_data)
    wcss_log = np.log(compute_wcss(ref_data, labels_ref))
    return (k, b_idx, wcss_log)

In [ ]:
# It's helpful to run this function using parallel computing to speed things up.
def compute_gap_statistic_with_parallel(X, k_values, reference_data_list, n_jobs=-1):
    B = len(reference_data_list)
    ref_tasks = [(k, ref_data, b) for b, ref_data in enumerate(reference_data_list) for k in k_values]

    # Parallel computation of reference WCSS
    results = Parallel(n_jobs=n_jobs)(
        delayed(cluster_reference_data)(k, ref_data, b) for (k, ref_data, b) in tqdm(ref_tasks, desc="Clustering reference data")
    )

    # Build dictionary: ref_wcss_dict[(k, b)] = log(WCSS)
    ref_wcss_dict = {(k, b): wcss_log for (k, b, wcss_log) in results}

    # Compute real WCSS and Gap values
    WCSS_scores = []
    gap_values = []

    for k in tqdm(k_values, desc="Evaluating real data"):
        model = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels = model.fit_predict(X)
        wcss_real = compute_wcss(X, labels)
        WCSS_scores.append(wcss_real)

        ref_log_wcss = [ref_wcss_dict[(k, b)] for b in range(B)]
        gap_k = np.mean(ref_log_wcss) - np.log(wcss_real)
        gap_values.append(gap_k)

    return WCSS_scores, gap_values

In [ ]:
# Once this code block is executed, you can plot the outputs to determine the optimal k.
reference_data_list = generate_reference_data(X, 10)
WCSS_scores, gap_values = compute_gap_statistic_with_parallel(X, k_values, reference_data_list, n_jobs=-1)

### <font color="black">**Creating the Final Clusters**</font>

After settling on a viable option of k, perform the final clustering and export the relevant data objects.

In [ ]:
clustering = AgglomerativeClustering(
    n_clusters=40,
    linkage='ward',
    metric='euclidean'
)

clustering.fit(X)

In [ ]:
cluster_labels = clustering.labels_

cluster_results = pd.DataFrame({
    'patient_id': patient_ids,
    'cluster': cluster_labels
})

cluster_patient_ids = cluster_results.groupby('cluster')['patient_id'].apply(list).to_dict()

Now I want to sort the original clusters by their patient counts (largest → smallest), then create a new dictionary where cluster IDs are reassigned so that Cluster 1 is the largest group, Cluster 2 is the next largest, and so on.

In [ ]:
sorted_clusters = sorted(cluster_patient_ids.items(), key=lambda x: len(x[1]), reverse=True)

sorted_cluster_patient_ids = {} # This is now our main dictionary representing our clusters
for i, (_, patients) in enumerate(sorted_clusters):
    sorted_cluster_patient_ids[i + 1] = patients

sorted_cluster_patient_ids.keys()

In [ ]:
# Export using pickle
with open("sorted_cluster_patient_ids.pkl", "wb") as f:
    pickle.dump(sorted_cluster_patient_ids, f)

#### <font color="black">**Creating and Exporting Treatment Groupings of the Clusters**</font>

Before applying the procedures described below, it is necessary to review the clustering results and determine meaningful groupings. In our workflow, this involved both statistical and exploratory analyses, supplemented by feedback from endocrinologists and other domain experts.

In [ ]:
# Early Dropout
Early_Dropout_Group = [1, 2, 3, 4, 5, 9, 11, 15, 17, 19]

# Variant Therapies
Variant_Therapies = [13, 16, 21]

# Monotherapy
Monotherapy_Group = [6, 7, 8, 18, 29]

# Dual Therapy
Dual_Therapy_Group = [12, 23, 25, 28, 31, 32, 36, 37]

# Complex Therapy
Complex_Therapy_Group = [22, 34, 40]

#GLP-1 Therapy
GLP1_Group = [10, 14, 20, 24, 26, 27, 30, 33, 35, 38, 39]

In [ ]:
monotherapy_patients = {}

for cluster_id, patients in sorted_cluster_patient_ids.items():
    if cluster_id in Monotherapy_Group:
        monotherapy_patients[cluster_id] = patients

In [ ]:
dual_therapy_patients = {}

for cluster_id, patients in sorted_cluster_patient_ids.items():
    if cluster_id in Dual_Therapy_Group:
        dual_therapy_patients[cluster_id] = patients

In [ ]:
complex_therapy_patients = {}

for cluster_id, patients in sorted_cluster_patient_ids.items():
    if cluster_id in Complex_Therapy_Group:
        complex_therapy_patients[cluster_id] = patients

In [ ]:
GLP_1_therapy_patients = {}

for cluster_id, patients in sorted_cluster_patient_ids.items():
  if cluster_id in GLP1_Group:
    GLP_1_therapy_patients[cluster_id] = patients

In [ ]:
variant_therapy_patients = {}

for cluster_id, patients in sorted_cluster_patient_ids.items():
  if cluster_id in Variant_Therapies:
    variant_therapy_patients[cluster_id] = patients

In [ ]:
early_dropout_patients = {}

for cluster_id, patients in sorted_cluster_patient_ids.items():
  if cluster_id in Early_Dropout_Group:
    early_dropout_patients[cluster_id] = patients

Export using pickle

In [ ]:
with open("monotherapy_patients.pkl", "wb") as f:
    pickle.dump(monotherapy_patients, f)

with open("dual_therapy_patients.pkl", "wb") as f:
    pickle.dump(dual_therapy_patients, f)

with open("complex_therapy_patients.pkl", "wb") as f:
    pickle.dump(complex_therapy_patients, f)

with open("GLP_1_therapy_patients.pkl", "wb") as f:
    pickle.dump(GLP_1_therapy_patients, f)

with open("variant_therapy_patients.pkl", "wb") as f:
    pickle.dump(variant_therapy_patients, f)

with open("early_dropout_patients.pkl", "wb") as f:
    pickle.dump(early_dropout_patients, f)